# Day 12 — Cache Benchmark
**Author: Sheshikala**  
**Date: 2026-03-12**

Comparing 3 caching patterns:
- **Cache-Aside** (Lazy Loading)
- **Write-Through**
- **Write-Behind** (Write-Back)

Before running: make sure MongoDB and Redis are running
```
docker run -d -p 27017:27017 --name mongodb mongo
docker run -d -p 6379:6379 --name redis redis
```

In [ ]:
import redis
import json
import time
import random
import matplotlib.pyplot as plt
from pymongo import MongoClient

r      = redis.Redis(host='localhost', port=6379, db=2, decode_responses=True)
client = MongoClient('mongodb://localhost:27017')
db     = client['benchmark_db']
col    = db['ml_experiments']

# seed 100 experiments into MongoDB
col.drop()
models = ['BERT', 'RoBERTa', 'DistilBERT', 'XGBoost', 'LightGBM']
experiments = [
    {
        '_id'   : i,
        'name'  : f'Experiment_{i}',
        'model' : random.choice(models),
        'f1'    : round(random.uniform(0.82, 0.96), 4),
        'status': random.choice(['completed', 'completed', 'running'])
    }
    for i in range(1, 101)
]
col.insert_many(experiments)
print(f'Seeded {col.count_documents({})} experiments')
print(f'Redis connected: {r.ping()}')

## Pattern 1 — Cache-Aside (Lazy Loading)

How it works:
1. Check cache first
2. If MISS → fetch from MongoDB → store in cache → return
3. If HIT → return directly from Redis

Best for: read-heavy workloads where not all data is needed at once

In [ ]:
CACHE_TTL = 60

def cache_aside_get(exp_id):
    key    = f'ca:exp:{exp_id}'
    cached = r.get(key)
    if cached:
        return json.loads(cached), 'HIT'
    # MISS - go to MongoDB
    doc = col.find_one({'_id': exp_id}, {'_id': 0})
    r.setex(key, CACHE_TTL, json.dumps(doc))
    return doc, 'MISS'

# clear old keys
for k in r.keys('ca:*'): r.delete(k)

# benchmark: 200 reads on IDs 1-20 (lots of repeat = lots of hits)
ids           = [random.randint(1, 20) for _ in range(200)]
times_ca      = []
hits_ca       = 0
misses_ca     = 0

for eid in ids:
    t0         = time.perf_counter()
    _, status  = cache_aside_get(eid)
    times_ca.append((time.perf_counter() - t0) * 1000)
    if status == 'HIT': hits_ca += 1
    else:               misses_ca += 1

hit_rate_ca = hits_ca / len(ids) * 100
avg_ca      = sum(times_ca) / len(times_ca)

print(f'Cache-Aside: hits={hits_ca}, misses={misses_ca}, hit_rate={hit_rate_ca:.1f}%')
print(f'Avg latency = {avg_ca:.3f} ms')

## Pattern 2 — Write-Through

How it works:
1. Write to MongoDB AND Redis at the same time
2. Cache is always up to date

Best for: read-heavy apps where stale data is not acceptable

In [ ]:
def write_through_update(exp_id, new_f1):
    key = f'wt:exp:{exp_id}'
    # write to MongoDB first
    col.update_one({'_id': exp_id}, {'$set': {'f1': new_f1}})
    # immediately update Redis too
    doc = col.find_one({'_id': exp_id}, {'_id': 0})
    r.setex(key, CACHE_TTL, json.dumps(doc))

def write_through_get(exp_id):
    key    = f'wt:exp:{exp_id}'
    cached = r.get(key)
    if cached: return json.loads(cached), 'HIT'
    doc = col.find_one({'_id': exp_id}, {'_id': 0})
    r.setex(key, CACHE_TTL, json.dumps(doc))
    return doc, 'MISS'

for k in r.keys('wt:*'): r.delete(k)

write_times_wt = []
read_times_wt  = []

for i in range(1, 101):
    t0 = time.perf_counter()
    write_through_update(i, round(random.uniform(0.82, 0.96), 4))
    write_times_wt.append((time.perf_counter() - t0) * 1000)

for eid in [random.randint(1, 100) for _ in range(100)]:
    t0 = time.perf_counter()
    write_through_get(eid)
    read_times_wt.append((time.perf_counter() - t0) * 1000)

avg_write_wt = sum(write_times_wt) / len(write_times_wt)
avg_read_wt  = sum(read_times_wt)  / len(read_times_wt)
print(f'Write-Through: avg write={avg_write_wt:.3f}ms | avg read={avg_read_wt:.3f}ms')

## Pattern 3 — Write-Behind (Write-Back)

How it works:
1. Write to Redis immediately (fast response)
2. MongoDB write happens later in a batch

Best for: write-heavy workloads where small delay is acceptable

In [ ]:
DIRTY_SET = 'wb:dirty'

def write_behind_update(exp_id, new_f1):
    key = f'wb:exp:{exp_id}'
    r.setex(key, CACHE_TTL, json.dumps({'f1': new_f1, 'exp_id': exp_id}))
    r.sadd(DIRTY_SET, exp_id)   # mark as needing DB sync

def flush_to_db():
    dirty = r.smembers(DIRTY_SET)
    for eid in dirty:
        key    = f'wb:exp:{int(eid)}'
        cached = r.get(key)
        if cached:
            data = json.loads(cached)
            col.update_one({'_id': int(eid)}, {'$set': {'f1': data['f1']}})
            r.srem(DIRTY_SET, eid)
    return len(dirty)

for k in r.keys('wb:*'): r.delete(k)

write_times_wb = []
for i in range(1, 101):
    t0 = time.perf_counter()
    write_behind_update(i, round(random.uniform(0.82, 0.96), 4))
    write_times_wb.append((time.perf_counter() - t0) * 1000)

avg_write_wb = sum(write_times_wb) / len(write_times_wb)
print(f'Write-Behind: avg write={avg_write_wb:.3f}ms (cache only, no DB yet)')

t0         = time.perf_counter()
flushed    = flush_to_db()
flush_time = (time.perf_counter() - t0) * 1000
print(f'Flushed {flushed} records to MongoDB in {flush_time:.1f}ms')

## Results Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Day 12 — Cache Pattern Benchmark (Sheshikala)', fontsize=14, fontweight='bold')

# plot 1: hit vs miss latency for cache-aside
miss_times = times_ca[:misses_ca]
hit_times  = times_ca[misses_ca:]
axes[0].boxplot([miss_times, hit_times], labels=['Cache MISS\n(MongoDB)', 'Cache HIT\n(Redis)'])
axes[0].set_title('Cache-Aside\nHit vs Miss Latency')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_facecolor('#f8f9fa')

# plot 2: write latency comparison
patterns = ['Write-Through\n(DB + Cache)', 'Write-Behind\n(Cache only)']
avgs     = [avg_write_wt, avg_write_wb]
colors   = ['#e74c3c', '#2ecc71']
bars = axes[1].bar(patterns, avgs, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, avgs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}ms', ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Write Latency Comparison')
axes[1].set_ylabel('Avg Write Time (ms)')
axes[1].set_facecolor('#f8f9fa')

# plot 3: hit rate pie
axes[2].pie(
    [hits_ca, misses_ca],
    labels=[f'HIT ({hits_ca})', f'MISS ({misses_ca})'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%',
    startangle=90
)
axes[2].set_title(f'Cache-Aside Hit Rate\n{hit_rate_ca:.1f}% over 200 reads')

plt.tight_layout()
plt.savefig('cache_benchmark_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as cache_benchmark_chart.png')

In [ ]:
print('=' * 65)
print(f'{"CACHE BENCHMARK SUMMARY":^65}')
print('=' * 65)
print(f'{"Pattern":<20} {"Avg Latency":<20} {"Consistency":<15} {"Best For"}')
print('-' * 65)
rows = [
    ('Cache-Aside',   f'{avg_ca:.3f}ms (read)',       'Eventual', 'Read-heavy, sparse'),
    ('Write-Through', f'{avg_write_wt:.3f}ms (write)', 'Strong',   'Always fresh data'),
    ('Write-Behind',  f'{avg_write_wb:.3f}ms (write)', 'Eventual', 'Write-heavy apps'),
]
for name, lat, cons, best in rows:
    print(f'{name:<20} {lat:<20} {cons:<15} {best}')
print('=' * 65)
speedup = avg_write_wt / avg_write_wb
print(f'Write-Behind is {speedup:.1f}x faster than Write-Through for writes')
print(f'Cache-Aside hit rate: {hit_rate_ca:.1f}%')
print('\n✅ cache_benchmark.ipynb completed!')